In [ ]:
import os

os.chdir("/Users/markusgambietz/PhD/01_Python_Projects/biosym")
import jax

import biosym
from biosym.model.model import load_model
from biosym.utils import *

In [ ]:
# Create a new model instance
model = load_model("tests/models/gait2d_torque/gait2d_torque.yaml")
input_vector = biosym.utils.states.stack_dataclasses([model.default_inputs] * 64)

In [ ]:
from functools import partial

import jax.numpy as jnp

mappable_confun = partial(model.run["confun"], constants=input_vector[0].constants)
jax.vmap(mappable_confun, in_axes=(0))(input_vector.states).shape
mappable_grffun = partial(model.run["gc_model"], constants=input_vector[0].constants)


@jax.jit
def eom_confun(states):
    eom = jax.vmap(mappable_confun, in_axes=(0))(states)
    return jnp.sum(jnp.abs(eom))


@jax.jit
def grf_confun(states):
    grf, mom = jax.vmap(mappable_grffun, in_axes=(0))(states)
    grfviol = (
        grf.reshape(-1, 6) - states.model[:, model.ext_forces["idx"] : model.ext_forces["idx"] + model.ext_forces["n"]]
    )
    momviol = (
        mom.reshape(-1, 6)
        - states.model[:, model.ext_torques["idx"] : model.ext_torques["idx"] + model.ext_torques["n"]]
    )
    return jnp.sum(jnp.abs(grfviol)) + jnp.sum(jnp.abs(momviol))


@jax.jit
def euler_confun(states):
    pos = states.model[:, model.coordinates["idx"] : model.coordinates["idx"] + model.coordinates["n"]]
    vel = states.model[:, model.speeds["idx"] : model.speeds["idx"] + model.speeds["n"]]
    acc = states.model[:, model.accs["idx"] : model.accs["idx"] + model.accs["n"]]
    # Somewhat symplectic Euler integration error

    e_pos = jnp.diff(pos, axis=0) * 100 - (vel[:-1])
    e_vel = jnp.diff(vel, axis=0) * 100 - (acc[1:])
    return jnp.sum(jnp.abs(e_pos)) + jnp.sum(jnp.abs(e_vel))


@jax.jit
def opt_torques(states):
    torques = states.model[:, model.forces["idx"] : model.forces["idx"] + model.forces["n"]]
    return jnp.sum(jnp.abs(torques))


@jax.jit
def task_constraint(task, res):
    mean, std = task[..., :45], task[..., 45:]
    states_error = (res - mean) / std
    return jnp.sum(jnp.abs(states_error))

In [ ]:
from dataclasses import dataclass
from typing import Tuple

import optax
from flax import linen as nn
from flax.training import train_state
from jax import random

from notebooks.network import *

# autoreload
%load_ext autoreload
%autoreload 2


def compute_metrics(res, task, w):
    states = biosym.utils.states.States(
        model=res.reshape(-1, res.shape[-1]),  # Flatten batch and sequence dimensions
        gc_model=jnp.zeros((res.shape[0] * res.shape[1], 0)),
        actuator_model=jnp.zeros((res.shape[0] * res.shape[1], 0)),
        h=jnp.zeros((res.shape[0] * res.shape[1], 0)),
    )
    eom = eom_confun(states) * w[0]
    grf = grf_confun(states) * w[1]
    eul = euler_confun(states) * w[2]
    opt = opt_torques(states) * w[3]
    task = task_constraint(task, res) * w[4]

    return {"eom": eom, "grf": grf, "eul": eul, "opt": opt, "task": task}


w = jnp.ones(5)


# ---------------------------
# Train state
# ---------------------------
@dataclass
class TrainConfig:
    learning_rate: float = 1e-3
    weight_decay: float = 1e-5
    seed: int = 0


def create_train_state(rng, model: nn.Module, input_shape: Tuple[int, int, int], cfg: TrainConfig):
    params = model.init(rng, jnp.zeros(input_shape), train=True)["params"]
    tx = optax.adamw(learning_rate=cfg.learning_rate, weight_decay=cfg.weight_decay)
    return train_state.TrainState.create(apply_fn=model.apply, params=params, tx=tx)


# ---------------------------
# Training & evaluation steps
# ---------------------------
@jax.jit
def train_step(state: train_state.TrainState, batch: dict):
    seqs = batch["seq"]

    def loss_fn(params):
        res = state.apply_fn({"params": params}, seqs, train=True)
        states = biosym.utils.states.States(
            model=res.reshape(-1, res.shape[-1]),  # Flatten batch and sequence dimensions
            gc_model=jnp.zeros((res.shape[0] * res.shape[1], 0)),
            actuator_model=jnp.zeros((res.shape[0] * res.shape[1], 0)),
            h=jnp.zeros((res.shape[0] * res.shape[1], 0)),
        )
        eom = eom_confun(states) * w[0]
        grf = grf_confun(states) * w[1]
        eul = euler_confun(states) * w[2]
        opt = opt_torques(states) * w[3]
        task = task_constraint(seqs, res) * w[4]
        return eom + grf + eul + opt + task, res

    (loss, res), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    metrics = compute_metrics(res, seqs, w) | {"loss": loss}
    return state, metrics


@jax.jit
def eval_step(state: train_state.TrainState, batch: dict):
    seqs = batch["seq"]
    logits = state.apply_fn({"params": state.params}, seqs, train=False)
    return compute_metrics(logits, seqs, w)


# ---------------------------
# Toy dataset utilities
# ---------------------------
def make_toy_batch(rng, batch_size: int, length: int = 500, channels: int = 45) -> dict:
    key_img, _ = random.split(rng)

    def sample_one(key):
        # shape (length, channels) -> (500, 45)
        seq = random.normal(key, (length, 2 * channels))
        seq = seq.at[:, 45:].set(random.uniform(key, (length, channels), minval=1e2, maxval=1e5))
        seq = seq.at[:, 45].set(0.5)
        speed = random.uniform(key, (1,), minval=0.5, maxval=2.0)
        seq = seq.at[:, 0].set(jnp.arange(length) / 100 * speed)
        # mask depends only on length axis, keep channel dim = 1
        mask = (seq.mean(axis=-1, keepdims=True) > 0).astype(jnp.float32)  # (length, 1)
        return seq, mask

    seqs = []
    for i in range(batch_size):
        s, m = sample_one(random.fold_in(key_img, i))
        seqs.append(s)
    return {"seq": jnp.stack(seqs)}


# ---------------------------
# Training loop demo
# ---------------------------
def train_demo(num_steps: int = 200, batch_size: int = 4, state: train_state.TrainState = None):
    cfg = TrainConfig()
    rng = random.PRNGKey(cfg.seed)
    model = UNet1D(base_features=32, num_classes=1)
    input_shape = (batch_size, 500, 90)  # (batch, length, channels)
    if state is None:
        state = create_train_state(rng, model, input_shape, cfg)

    metrics_history = {"eom": [], "grf": [], "eul": [], "opt": [], "task": []}

    for step in tqdm(range(1, num_steps + 1), desc="Training"):
        rng, key = random.split(rng)
        batch = make_toy_batch(key, batch_size=batch_size, length=500, channels=45)
        state, metrics = train_step(state, batch)
        for k in metrics_history:
            metrics_history[k].append(float(metrics[k]))
        if step % 20 == 0:
            eval_metrics = eval_step(state, batch)
            print(f"Step {step:04d} | Eval Metrics: {eval_metrics}")
            for k in metrics_history:
                metrics_history[k].append(float(eval_metrics[k]))

    return state, metrics_history

In [ ]:
state, metrics_history = train_demo(num_steps=200, batch_size=16)

In [ ]:
state, metrics_history = train_demo(num_steps=200, batch_size=16, state=state)